### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [19]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path
# from langchain.text_splitter import RecursiveCharacterTextSplitter  ""not working with latest version of langchain, so using langchain_classic""
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

In [20]:
### Read all the PDF files in the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files in the directory
    pdf_files=list[Path](pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file.name}")
        try:
            #Load the PDF file using PyMuPDFLoader
            loader=PyMuPDFLoader(str(pdf_file))
            documents=loader.load()

            # Add source information to the metadata of each document
            for doc in documents:
                doc.metadata["source"]= pdf_file.name
                doc.metadata["author"]="Vidyanandk"
                doc.metadata["date_created"]=str(pdf_file.stat().st_ctime)
                doc.metadata["file_type"]="pdf"
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#Process all PDFs in the specified directory
pdf_directory="../data/pdf_files"
all_pdf_documents=process_all_pdfs(pdf_directory)


Found 6 PDF files in the directory.

Processing file: Performance Improvement Process Policy.pdf
Loaded 5 pages

Processing file: Asset-Laptop policy.pdf
Loaded 1 pages

Processing file: Amantya-InformationSecurityPolicy.pdf
Loaded 28 pages

Processing file: Amantya Rewards & Recognition Policy.pdf
Loaded 6 pages

Processing file: Exit Policy.pdf
Loaded 12 pages

Processing file: Compensation and Benefits Policy - V4.pdf
Loaded 10 pages

Total documents loaded: 62


In [21]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-01-27T17:24:27+05:30', 'source': 'Performance Improvement Process Policy.pdf', 'file_path': '../data/pdf_files/Performance Improvement Process Policy.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Vidyanandk', 'subject': '', 'keywords': '', 'moddate': '2023-01-27T17:24:27+05:30', 'trapped': '', 'modDate': "D:20230127172427+05'30'", 'creationDate': "D:20230127172427+05'30'", 'page': 0, 'date_created': '1779437571.4402757', 'file_type': 'pdf'}, page_content='© Copyright AMANTYA TECHNOLOGIES \nwww.amantyatech.com \n \n \n \n \n \nPERFORMANCE IMPROVEMENT PLAN POLICY \n \n \nDate: 24-Dec-2022 \nPrepared By \nPallavi Mishra \nDate: 27-Jan-2023 \nReviewed By \n \nRajiv Agarwal \nDate: 27-Jan-2023 \nApproved By  \nAnuradha Gupta (CEO) \n \nVersion \nDate \nDescription \n1.0 \n01-Jun-2022 \nCreation of Policy \n1.1 \n30-Dec-2022 \nReview of the Policy \n \n \n \

In [22]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks of specified size with overlap for Rag performance"""
    text_Splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_Splitter.split_documents(documents)
    print(f"Split into {len(documents)} documents and {len(split_docs)} chunks")

    #Shown the first chunk and its metadata as an example
    if split_docs:
        print(f"\nexample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print the first 500 characters of the chunk
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [23]:
chunks=split_documents(all_pdf_documents)
chunks

Split into 62 documents and 171 chunks

example chunk:
Content: © Copyright AMANTYA TECHNOLOGIES 
www.amantyatech.com 
 
 
 
 
 
PERFORMANCE IMPROVEMENT PLAN POLICY 
 
 
Date: 24-Dec-2022 
Prepared By 
Pallavi Mishra 
Date: 27-Jan-2023 
Reviewed By 
 
Rajiv Agarwa...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-01-27T17:24:27+05:30', 'source': 'Performance Improvement Process Policy.pdf', 'file_path': '../data/pdf_files/Performance Improvement Process Policy.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Vidyanandk', 'subject': '', 'keywords': '', 'moddate': '2023-01-27T17:24:27+05:30', 'trapped': '', 'modDate': "D:20230127172427+05'30'", 'creationDate': "D:20230127172427+05'30'", 'page': 0, 'date_created': '1779437571.4402757', 'file_type': 'pdf'}


Split into 62 documents and 171 chunks

example chunk:
Content: © Copyright AMANTYA TECHNOLOGIES 
www.amantyatech.com 
 
 
 
 
 
PERFORMANCE IMPROVEMENT PLAN POLICY 
 
 
Date: 24-Dec-2022 
Prepared By 
Pallavi Mishra 
Date: 27-Jan-2023 
Reviewed By 
 
Rajiv Agarwa...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-01-27T17:24:27+05:30', 'source': 'Performance Improvement Process Policy.pdf', 'file_path': '../data/pdf_files/Performance Improvement Process Policy.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Vidyanandk', 'subject': '', 'keywords': '', 'moddate': '2023-01-27T17:24:27+05:30', 'trapped': '', 'modDate': "D:20230127172427+05'30'", 'creationDate': "D:20230127172427+05'30'", 'page': 0, 'date_created': '1779437571.4402757', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-01-27T17:24:27+05:30', 'source': 'Performance Improvement Process Policy.pdf', 'file_path': '../data/pdf_files/Performance Improvement Process Policy.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Vidyanandk', 'subject': '', 'keywords': '', 'moddate': '2023-01-27T17:24:27+05:30', 'trapped': '', 'modDate': "D:20230127172427+05'30'", 'creationDate': "D:20230127172427+05'30'", 'page': 0, 'date_created': '1779437571.4402757', 'file_type': 'pdf'}, page_content='© Copyright AMANTYA TECHNOLOGIES \nwww.amantyatech.com \n \n \n \n \n \nPERFORMANCE IMPROVEMENT PLAN POLICY \n \n \nDate: 24-Dec-2022 \nPrepared By \nPallavi Mishra \nDate: 27-Jan-2023 \nReviewed By \n \nRajiv Agarwal \nDate: 27-Jan-2023 \nApproved By  \nAnuradha Gupta (CEO) \n \nVersion \nDate \nDescription \n1.0 \n01-Jun-2022 \nCreation of Policy \n1.1 \n30-Dec-2022 \nReview of the Policy \n \n \n \

### Embedding And vectorStoreDb

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [40]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of strings to embed

        Returns:
           numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise e
    
## initialize the embedding manager 
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1669.64it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_179258/2297539402.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [41]:
class vectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_chunks", persist_directory: str = "../data/chroma_db"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection to use
            persist_directory: Directory where ChromaDB will persist data
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client=None
        self.collection=None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # create the persistence directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)
            print(f"Initializing ChromaDB client with persistence at: {self.persist_directory}...")
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection for storing document embeddings
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "Collection for PDF document chunks and their embeddings"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=vectorStore()
vectorstore


Initializing ChromaDB client with persistence at: ../data/chroma_db...
Vector store initialized. Collection: pdf_chunks
Existing documents in collection: 0


In [32]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-01-27T17:24:27+05:30', 'source': 'Performance Improvement Process Policy.pdf', 'file_path': '../data/pdf_files/Performance Improvement Process Policy.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Vidyanandk', 'subject': '', 'keywords': '', 'moddate': '2023-01-27T17:24:27+05:30', 'trapped': '', 'modDate': "D:20230127172427+05'30'", 'creationDate': "D:20230127172427+05'30'", 'page': 0, 'date_created': '1779437571.4402757', 'file_type': 'pdf'}, page_content='© Copyright AMANTYA TECHNOLOGIES \nwww.amantyatech.com \n \n \n \n \n \nPERFORMANCE IMPROVEMENT PLAN POLICY \n \n \nDate: 24-Dec-2022 \nPrepared By \nPallavi Mishra \nDate: 27-Jan-2023 \nReviewed By \n \nRajiv Agarwal \nDate: 27-Jan-2023 \nApproved By  \nAnuradha Gupta (CEO) \n \nVersion \nDate \nDescription \n1.0 \n01-Jun-2022 \nCreation of Policy \n1.1 \n30-Dec-2022 \nReview of the Policy \n \n \n \

In [42]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]

###Generate embeddings for the document chunks
embeddings=embedding_manager.generate_embeddings(texts)

### Add the chunks and their embeddings to the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 171 texts...


Batches: 100%|██████████| 6/6 [00:05<00:00,  1.04it/s]


Generated embeddings with shape: (171, 384)
Adding 171 documents to vector store...
Successfully added 171 documents to vector store
Total documents in collection: 171


Retriever PipeLine from VectorStore

In [47]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: vectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


                



In [44]:
rag_retriever

In [51]:
rag_retriever.retrieve(" Employee Welfare and Benefits", top_k=3, score_threshold=0.0)

Retrieving documents for query: ' Employee Welfare and Benefits'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 108.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_d875f1bd_152',
  'content': '6. \nEmployee Welfare and Benefits: ......................................................................................................... 8 \n6.1 \nLeave Entitlement: - .................................................................................................................... 8 \n6.2 \nGroup Health Insurance............................................................................................................... 9 \n6.3 \nGroup Term Life Insurance .......................................................................................................... 9 \n6.4 \nRetirement Benefits..................................................................................................................... 9 \n6.5 \nCar Lease Benefits ....................................................................................................................... 9 \n6.6',
  'metadata': {'creator': 'Microsoft® Word for Microsoft 365',
   'trapped'

In [52]:
rag_retriever.retrieve("Define exit policy")

Retrieving documents for query: 'Define exit policy'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.16it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_463afbff_121',
  'content': 'Exit Policy \nVersion 1.3',
  'metadata': {'date_created': '1779437537.1629586',
   'total_pages': 12,
   'format': 'PDF 1.7',
   'title': 'Network in a box',
   'moddate': '2026-03-02T09:49:47+05:30',
   'doc_index': 121,
   'creator': 'Microsoft® Word 2019',
   'modDate': "D:20260302094947+05'30'",
   'keywords': '',
   'producer': 'Microsoft® Word 2019',
   'trapped': '',
   'file_path': '../data/pdf_files/Exit Policy.pdf',
   'page': 0,
   'content_length': 24,
   'source': 'Exit Policy.pdf',
   'subject': '',
   'creationDate': "D:20260302094947+05'30'",
   'creationdate': '2026-03-02T09:49:47+05:30',
   'file_type': 'pdf',
   'author': 'Vidyanandk'},
  'similarity_score': 0.6272822618484497,
  'distance': 0.3727177381515503,
  'rank': 1},
 {'id': 'doc_3b48aceb_126',
  'content': 'Copyright 2023 © Amantya Technologies \n3 \n \n \n \n1. Purpose \n \nThe purpose of this policy is to outline the process of the exit at Amantya Technologies, in

### RAG Pipline- VectorDB to LLM Output Generation

In [54]:
import os
from dotenv import load_dotenv
load_dotenv()

print("Environment variables loaded successfully.")
# print(os.getenv("GROQ_API_KEY"))

Environment variables loaded successfully.


In [67]:
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage, AIMessage
from langchain.prompts import PromptTemplate

ModuleNotFoundError: No module named 'langchain.schema'

In [66]:
class GroqLLM:
    """Handles interactions with the Groq LLM for generating responses based on retrieved documents"""
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str = None):
        """
        Initialize the Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: API key for authenticating with Groq (optional, can also be set via environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key must be provided either as an argument or via the GROQ_API_KEY environment variable.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized GroqLLM with model: {self.model_name}")
    
    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
            Generate response using retrieved context
            
            Args:
                query: User question
                context: Retrieved document context
                max_length: Maximum response length
                
            Returns:
                Generated response string
        """

        # Create a prompt template for the LLM
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

            Context:
            {context}

            Question: {question}

            Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        # format the prompt with the retrieved context and user query
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response fromt the LLM
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        
        except Exception as e:
            print(f"Error generating response from Groq LLM: {e}")
            return "Sorry, I encountered an error while generating the response."
    
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

        Question: {query}

        Answer:
        """
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [59]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized GroqLLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [60]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### Integration Vectordb Context pipeline With LLM output

In [64]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [65]:
answer=rag_simple("What steps are needed during exit?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What steps are needed during exit?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


According to the Exit Policy, the following steps are needed during exit:

1. The exit interview shall be carried out by an HR team member in respect of the exiting employee.
2. The proceedings of the exit interview shall be recorded in the 'Exit Interview Form' for documentation and record keeping purposes.


### Enhanced RAG Pipeline features

In [68]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("ORGANIZATION FOR INFORMATION SECURITY POLICY", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'ORGANIZATION FOR INFORMATION SECURITY POLICY'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The objective of this section is to define the organization structure along with roles & responsibilities for the implementation of the Information Security Policy and Information Security Management System.
Sources: [{'source': 'Amantya-InformationSecurityPolicy.pdf', 'page': 9, 'score': 0.4284043312072754, 'preview': 'www.amantyatech.com \n© Copyright Amantya Technologies Private Limited 2019-25 \n \n \n \n \n \n \n \n \n2.6 Information Security Policy Implementation Exception Management  \n \nIn case there are exceptions to the implementation of the information security policy, due to lack of \nfeasibility of implementation ...'}, {'source': 'Amantya-InformationSecurityPolicy.pdf', 'page': 5, 'score': 0.3967827558517456, 'preview': '3 \nOrganization of \nInformation Security \nDeals with management buy-in for information security, \nallocation of information security roles and responsibilities, \nand management of information security within Amantya \n4 \nAcceptable Use \nPo

In [71]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("ORGANIZATION FOR INFORMATION SECURITY POLICY", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\n\nFinal Answer:", result['answer'])
print("\nSummary:", result['summary'])
print("\nHistory:", result['history'][-1])

Retrieving documents for query: 'ORGANIZATION FOR INFORMATION SECURITY POLICY'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
www.amantya

tech.com 
© Copyright Amantya Technologies Private Limited 2019-25 
 
 
 
 
 
 
 
 
2.6 Information Security Policy Implementation Exception Management  
 
In case there are exceptions to the implementation of the information security policy, due to lack of 
feasibility of implementation or unfavourable cost – benefit analysis, exception for implementation 
can be sought from IST along with risks involved and mitigation plan. IST shall have authority 
to approve such exceptions on case to case basis. 
 
3 ORGANIZATION FOR INFORMATION SECURITY POLICY 
3.1 Objective 
Objective of this section is to define organization structure along with roles & responsibilities for 
implementation of Information security policy and information security management system. 
3.2 Internal Organization 
Information Security will be given high priority in Amantya’s organization. Amantya will establish 
Information Security Team (IST) to review information security policy and monitor adoption of

3 
Organizat